# Homework 6 – Decision Trees & Random Forests
Predicting `fuel_efficiency_mpg` on the 2026 car fuel-efficiency dataset.

In [1]:
# Download the dataset (run once)
!wget -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

File ‘car_fuel_efficiency_2026.csv’ already there; not retrieving.



In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## Preparing the dataset

In [3]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [5]:
# Fill missing values with zeros
df = df.fillna(0)
df.isnull().sum().sum()

np.int64(0)

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.pop('fuel_efficiency_mpg').values
y_val = df_val.pop('fuel_efficiency_mpg').values
y_test = df_test.pop('fuel_efficiency_mpg').values

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [7]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

X_train.shape

(6000, 16)

## Question 1 – Decision tree with `max_depth=1`

In [8]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [9]:
split_feature = dv.get_feature_names_out()[dt.tree_.feature[0]]
# Map a one-hot feature like 'origin=USA' back to the original column
original_feature = split_feature.split('=')[0]
print('Split feature:', split_feature)
print('Original column:', original_feature)

Split feature: model_year
Original column: model_year


## Question 2 – Random forest, `n_estimators=10`

In [10]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print('Validation RMSE:', round(rmse, 4))

Validation RMSE: 1.8332


## Question 3 – Tuning `n_estimators`

In [11]:
scores = []
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores.append((n, round(rmse, 3)))

df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
df_scores

,n_estimators,rmse
0,10,1.833
1,50,1.774
2,100,1.770
3,150,1.769


In [12]:
best = df_scores.loc[df_scores.rmse.idxmin()]
print('Lowest RMSE at n_estimators =', int(best.n_estimators), '->', best.rmse)

Lowest RMSE at n_estimators = 150 -> 1.769


## Question 4 – Best `max_depth` (mean RMSE across `n_estimators`)

In [13]:
scores = []
for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        scores.append((d, n, rmse))

df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
df_scores.pivot(index='n_estimators', columns='max_depth', values='rmse').round(3)

max_depth,10,15,20,25
n_estimators,,,,
10,1.782,1.831,1.837,1.837
50,1.748,1.771,1.772,1.772
100,1.744,1.768,1.769,1.768
150,1.744,1.767,1.768,1.768


In [14]:
mean_rmse = df_scores.groupby('max_depth').rmse.mean()
print(mean_rmse.round(4))
print('\nBest max_depth:', mean_rmse.idxmin())

max_depth
10    1.7544
15    1.7842
20    1.7868
25    1.7863
Name: rmse, dtype: float64

Best max_depth: 10
